# Suite PAR — Parametrics

A parameter is a name a definition declares (`Schemas.OfParameter`), with a description: a symbol standing for what is
bound where the definition is referred to. Parameters are `Parametrics`', normal schemas and one extension, installed
before use ([CAP-112: parameters modeled as schema data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-112.md)): a set of parameters (`Parametrics.OfParameters`), each named by its entry
(`Parametrics.Defines`) and for a value of a schema or an object of an object schema, declared by a definition of any
kind (`parameters`). A schema declares parameters, and a reference object that the schema governs assigns a value to
each parameter for a value and binds each parameter for an object to an object, through the adjacencies
`Parametrics.extend` gives the schema ([CAP-314: parameters bound by the data objects their schema governs](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-314)). Here: installing it, declaring and
validating parameters, snapshots writing them, extending a schema, and assigning and binding parameters.

In [ ]:
import { Builders as B, JSON, Parametrics, Plain, Proxies, Schemas as S, Validators } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, entries_in, equal, given, map, raises, same, spec_of } from "./support.js";

<a id="PAR-01"></a>
## PAR-01 · Parametrics is installed before use: its schemas, a parameter's ends and every kind's set of parameters; installed again, it writes nothing

In [ ]:
{
  const KINDS = ["OfScalar", "OfObject", "OfUnion", "OfIntersection", "OfContainer", "OfAny", "OfRelation", "OfPattern",
    "OfExpression", "OfCollector", "OfModule"];
  const installed: any = new B.Store();
  assert(!installed.names().includes("Parametrics.OfParameters")); // not installed: a store has none
  assert(!installed.Schemas.OfParameter.as_schema().adjacencies.has("for_value"));
  assert(!installed.Schemas.OfObject.as_schema().adjacencies.has("parameters")); // the core declares none
  installed.transaction(Parametrics.install);
  assert(same(installed.names().filter((name: string) => name.startsWith("Parametrics.")), [
    "Parametrics.OfParameters", "Parametrics.Defines", "Parametrics.ForValue", "Parametrics.ForObject",
    "Parametrics.HasParameters", "Parametrics.AssignsValue", "Parametrics.BindsObject"]));
  const [Defines, ForValue, ForObject, HasParameters, AssignsValue, BindsObject] = ["Defines", "ForValue", "ForObject",
    "HasParameters", "AssignsValue", "BindsObject"].map((name) => installed.registered(`Parametrics.${name}`));
  const OfParameters = installed.registered("Parametrics.OfParameters");
  assert(OfParameters.ref && same([...OfParameters.adjacencies.keys()], ["parameters"]));
  assert(same([...Defines.entry.links.keys()], ["parameters", "parameter"]) && same([...Defines.entry.properties.keys()], ["name"]));
  const sets = (relation: any) => relation.uniques.map((unique: Set<string>) => [...unique].sort());
  assert(equal(sets(Defines), [["name", "parameters"], ["parameter"]])); // keyed by name
  assert(equal(sets(ForValue), [["schema"]]) && equal(sets(ForObject), [["schema"]])); // one schema per parameter, shared by many
  assert(equal(sets(HasParameters), [["defn"], ["params"]])); // unique in both directions
  assert(same([...AssignsValue.entry.links.keys()], ["object", "param"])
    && AssignsValue.entry.properties.get("value").schema.schemas.length === 0); // any value
  assert(same([...BindsObject.entry.links.keys()], ["object", "param", "obj_ref"])); // an object binds, not a schema
  const OfParameter = installed.Schemas.OfParameter.as_schema(); // the extension
  assert(equal([...OfParameter.adjacencies].slice(-2).map(([name, a]: any) => [name, a.relation, a.me]),
    [["for_value", ForValue, "param"], ["for_object", ForObject, "param"]]));
  assert(equal(OfParameter.anonymous.slice(-3).map((a: any) => [a.relation, a.me]), [
    [Defines, "parameter"], [AssignsValue, "param"], [BindsObject, "param"]])); // what defines it, and what binds it
  for (const kind of KINDS) {
    const held = installed.Schemas[kind].as_schema().adjacencies.get("parameters");
    assert(held.relation === HasParameters && held.me === "defn");
  }
  const log = JSON.dumps(Plain.ToPlain(installed).Snapshot());
  installed.transaction(Parametrics.install); // installed again: found, nothing written
  assert(JSON.dumps(Plain.ToPlain(installed).Snapshot()) === log);
  assert(!(new B.Store() as any).names().includes("Parametrics.OfParameters")); // each store's own
}

<a id="PAR-02"></a>
## PAR-02 · Every kind of schema declares a set of parameters, each named by its entry, described, and for a value of a schema or an object of an object schema

In [ ]:
/** A schema of every kind declaring a set holding the parameter n, for an int; Matrix, declaring rows and cols; and
 * Tree, whose parameter is for a reference to a tree. */
function declaring(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.declaring/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Object: S.OfObject, Relation: S.OfRelation, Union: S.OfUnion, Intersection: S.OfIntersection,
      Container: S.OfContainer, Pattern: S.OfPattern, Matrix: S.OfObject, Tree: S.OfObject })
    .spec([
      (_: any) => _.Object.name("par.Object").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
      (_: any) => _.Relation.name("par.Relation").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
      (_: any) => _.Union.name("par.Union").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]).branches([
        (t: any) => t.name("a").schema([S.OfStr]), (t: any) => t.name("b").schema([S.OfStr])]),
      (_: any) => _.Intersection.name("par.Intersection").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]).parts([
        (t: any) => t.name("a").schema([S.OfStr]), (t: any) => t.name("b").schema([S.OfStr])]),
      (_: any) => _.Container.name("par.Container").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]).index([S.OfInt]).element([S.OfStr]),
      (_: any) => _.Pattern.name("par.Pattern").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
      (_: any) => _.Matrix.name("Matrix").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        rows: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
        cols: (d: any) => d.parameter((p: any) => p.description("How many columns").for_value([S.OfInt])),
      }))]),
      (_: any) => _.Tree.name("Tree").ref(true).parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        root: (d: any) => d.parameter((p: any) => p.for_object([_.Tree])) }))]),
    ])]))]);
}

{
  const declared: any = new B.Store();
  declared.transaction(Parametrics.install);
  declared.transaction(declaring);
  for (const [name, kind] of [["par.Object", B.OfObject], ["par.Relation", B.OfRelation], ["par.Union", B.OfUnion],
    ["par.Intersection", B.OfIntersection], ["par.Container", B.OfContainer], ["par.Pattern", B.OfPattern]] as const) {
    const defn = declared.registered(name);
    const [held] = [...(new (kind.Builder as any)(defn, false, declared)).parameters]; // one set, its parameters keyed by their names
    assert(same([...held.params.parameters].map((e: any) => e.name.get()), ["n"]) && Parametrics.validate(declared, defn).length === 0);
  }
  const [matrix] = [...new B.OfObject.Builder(declared.registered("Matrix"), false, declared).parameters];
  const [rows, cols] = [...matrix.params.parameters]; // in order
  assert(rows.name.get() === "rows" && cols.name.get() === "cols" && cols.parameter instanceof S.OfParameter.Data);
  assert(rows.parameter.description === null && cols.parameter.description === "How many columns");
  const wide = () => new S.OfParameter.Data({ name: "w", description: "Its width" });
  assert(wide().equals(wide()) && !new S.OfParameter.Data({ name: "w" }).equals(wide())); // by what it holds
  assert(same([...new B.OfParameter.Builder(cols.parameter, false, declared).for_value].map((e: any) => e.schema.name.get()),
    ["Schemas.OfInt"]));
  const [tree] = [...new B.OfObject.Builder(declared.registered("Tree"), false, declared).parameters];
  const [root] = [...tree.params.parameters];
  const ends: any = new B.OfParameter.Builder(root.parameter, false, declared);
  assert(same([...ends.for_object].map((e: any) => e.schema.name.get()), ["Tree"]) && [...ends.for_value].length === 0); // for an object
  const log = JSON.dumps(Plain.ToPlain(declared).Snapshot());
  declared.transaction(declaring); // run again: found, nothing written
  assert(JSON.dumps(Plain.ToPlain(declared).Snapshot()) === log);
  const [Vector] = declared.transaction(() => { // imperatively, the set built where the definition links it
    const Vector = declared.Schemas.OfObject((o: any) => o.name("Vector").parameters([declared.Parametrics.OfParameters(
      (ps: any) => ps.parameters({ n: (d: any) => d.parameter((p: any) => p.for_value([declared.Schemas.OfInt])) }))]));
    return [Vector];
  });
  const [vector] = [...new B.OfObject.Builder(Vector, false, declared).parameters];
  assert(same([...vector.params.parameters].map((e: any) => e.name.get()), ["n"]) && Parametrics.validate(declared, Vector).length === 0);
  assert(Parametrics.validate(declared, declared.Schemas.OfInt.as_schema()).length === 0); // none declared: none to check
}

<a id="PAR-03"></a>
## PAR-03 · Validation checks a set's entries: reserved names, descriptions, a parameter for one of a value and an object, a pattern's names, and one set to one definition

In [ ]:
/** A pattern whose set has a reserved name, a parameter for neither a value nor an object, one for both, and names
 * its symbol and its variable have; and Shared and Twice, which share one set. */
function flawed(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.flawed/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Pattern: S.OfPattern, Node: S.OfObject })
    .spec([
      (_: any) => _.Node.name("par.Node").ref(true),
      (_: any) => _.Pattern.name("par.Flawed").symbols({ x: _.Node }).variables({ w: S.OfInt }).parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          $n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
          x: (d: any) => d.parameter((p: any) => p.for_object([_.Node])),
          w: (d: any) => d.parameter((p: any) => p), // for neither
          b: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt]).for_object([_.Node])), // for both
        }))]),
    ])]))]);
}

{
  const checked: any = new B.Store();
  checked.transaction(Parametrics.install);
  checked.transaction(flawed);
  const Flawed = checked.registered("par.Flawed");
  assert(same(Parametrics.validate(checked, Flawed), [
    "name '$n' is reserved: names starting with '$' belong to the wire format",
    "parameter 'w' is for neither a value nor an object: a parameter is for one",
    "parameter 'b' is for both a value and an object: a parameter is for one",
    "names used as both symbol and parameter: ['x']",
    "names used as both variable and parameter: ['w']"]));
  assert(Flawed.validate().length === 0); // a kind's own validate() does not read the extension's entries
  const [held] = [...new B.OfPattern.Builder(Flawed, false, checked).parameters];
  const dollar = [...held.params.parameters][0].parameter;
  dollar.description = true; // data the builders refuse to build
  assert(Parametrics.validate(checked, Flawed)[1] === "parameter '$n': a description is text, got bool");
  raises(TypeError, () => checked.transaction(() => checked.Schemas.OfParameter((p: any) => p.description(true))),
    "property 'description' holds str, got bool");
  const [One, Two] = checked.transaction(() => { // one set linked by two definitions, and a definition linking two sets
    const shared = checked.Parametrics.OfParameters((ps: any) => ps.parameters({
      n: (d: any) => d.parameter((p: any) => p.for_value([checked.Schemas.OfInt])) }));
    const other = checked.Parametrics.OfParameters((ps: any) => ps);
    const One = checked.Schemas.OfObject((o: any) => o.name("par.One").parameters([shared]));
    const Two = checked.Schemas.OfObject((o: any) => o.name("par.Two").parameters([shared, other]));
    return [One, Two];
  });
  assert(same(Parametrics.validate(checked, One), ["its set of parameters belongs to 2 definitions: a set belongs to one"]));
  assert(same(Parametrics.validate(checked, Two), ["it has 2 sets of parameters: a definition has one",
    "its set of parameters belongs to 2 definitions: a set belongs to one"]));
}

<a id="PAR-04"></a>
## PAR-04 · Snapshots write a definition's set as the extension's entry and read it back; a store without Parametrics ignores what needs it

In [ ]:
/** Grid, declaring rows and cols, each for an int, its cells a container of floats. */
function grid(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.grid/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Grid: S.OfObject })
    .spec([
      (_: any) => _.Grid.name("Grid").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        rows: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
        cols: (d: any) => d.parameter((p: any) => p.description("Its width").for_value([S.OfInt])),
      }))]).properties({ cells: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfFloat])) }),
    ])]))]);
}

{
  const written: any = new B.Store();
  written.transaction(Parametrics.install);
  written.transaction(grid);
  const log = Plain.ToPlain(written).Snapshot() as any[];
  const [entry] = entries_in(spec_of(log, "Grid", 1), "parameters"); // the extension's entry, with the data: written once
  assert(equal(entry, map({ params: { reference: { symbol: "0" } } })) && entries_in(spec_of(log, "Grid"), "parameters").length === 0);
  const parameters = entries_in(spec_of(log, "0", 1), "parameters"); // the set's, each defining the parameter its entry links
  assert(same(parameters.map((e) => e.get("name")), ["cols", "rows"])); // data's keyed entries, written in the order of their keys
  assert(equal(parameters.map((e) => given(e.get("parameter").get("def").get("mutations"))), [map({ description: "Its width" }), map({})]));
  const back: any = new B.Store();
  back.transaction(Parametrics.install); // a store with Parametrics installed
  Plain.FromPlain(back).Snapshot(log);
  assert(JSON.dumps(Plain.ToPlain(back).Snapshot()) === JSON.dumps(log)); // read back, written again: the same
  const [held] = [...new B.OfObject.Builder(back.registered("Grid"), false, back).parameters];
  assert(same([...held.params.parameters].map((e: any) => e.name.get()), ["cols", "rows"]));
  assert(Parametrics.validate(back, back.registered("Grid")).length === 0 && back.registered("Grid") !== written.registered("Grid"));
  back.transaction(grid); // its install, run again: found, nothing written
  assert(JSON.dumps(Plain.ToPlain(back).Snapshot()) === JSON.dumps(log));
  const bare: any = new B.Store(); // a store without it reads it as a snapshot made with extensions it lacks
  const ignored: any[] = [];
  Plain.FromPlain(bare).Snapshot(log, ignored);
  assert(same(ignored.map((i) => i.reason), [...Array(2).fill("Schemas.OfParameter has no adjacency 'for_value'"),
    "Schemas.OfObject has no adjacency 'parameters'"]));
  assert(same([...bare.registered("Grid").properties.keys()], ["cells"]) && !bare.Schemas.OfObject.as_schema().adjacencies.has("parameters"));
}

<a id="PAR-05"></a>
## PAR-05 · `extend` gives a reference object schema the adjacencies through which its objects assign and bind; it refuses a store without Parametrics, a value object schema and a member's name taken; run again, it writes nothing

In [ ]:
/** hw.Bus, a reference object declaring width, for an int, and host, for a hw.Chip; hw.Chip and hw.Spare, reference
 * objects whose schemas declare nothing; hw.Pin, a value object; hw.Taken and hw.Linked, which declare a member of each
 * name extend gives. */
function hardware(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.hardware/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Bus: S.OfObject, Chip: S.OfObject, Spare: S.OfObject, Pin: S.OfObject, Taken: S.OfObject,
      Linked: S.OfObject, Wires: S.OfRelation })
    .spec([
      (_: any) => _.Chip.name("hw.Chip").ref(true), // opts into nothing: any reference object may be bound
      (_: any) => _.Spare.name("hw.Spare").ref(true),
      (_: any) => _.Bus.name("hw.Bus").ref(true).properties({ label: S.OfStr }).parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          width: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
          host: (d: any) => d.parameter((p: any) => p.for_object([_.Chip])),
        }))]),
      (_: any) => _.Pin.name("hw.Pin").properties({ number: S.OfInt }),
      (_: any) => _.Wires.name("hw.Wires").links([(t: any) => t.name("from"), (t: any) => t.name("to")]),
      (_: any) => _.Taken.name("hw.Taken").ref(true).properties({ assigned_values: S.OfStr }),
      (_: any) => _.Linked.name("hw.Linked").ref(true).adjacencies({
        bound_objects: (a: any) => a.relation([_.Wires]).me("from") }),
    ])]))]);
}

const KINDS = ["OfScalar", "OfObject", "OfUnion", "OfIntersection", "OfContainer", "OfAny", "OfRelation", "OfPattern",
  "OfExpression", "OfCollector", "OfModule"];
const bare: any = new B.Store();
raises(LookupError, () => bare.transaction((t: any) => Parametrics.extend(t, [t.Schemas.OfObject])),
  "Parametrics.extend: the store has no Parametrics installed; install it first, "
  + "store.transaction(Parametrics.install)");
const hw: any = new B.Store();
hw.transaction(Parametrics.install);
hw.transaction(hardware);
const [Bus, AssignsValue, BindsObject] = ["hw.Bus", "Parametrics.AssignsValue", "Parametrics.BindsObject"]
  .map((name) => hw.registered(name));
assert(!Bus.adjacencies.has("assigned_values")); // not extended: its objects assign and bind nothing
hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Bus]));
assert(equal([...Bus.adjacencies].map(([name, a]: any) => [name, a.relation, a.me]), [
  ["assigned_values", AssignsValue, "object"], ["bound_objects", BindsObject, "object"]]));
{
  const log = JSON.dumps(Plain.ToPlain(hw).Snapshot());
  hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Bus, hw.registered("hw.Bus")])); // run again: nothing written
  assert(JSON.dumps(Plain.ToPlain(hw).Snapshot()) === log);
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Pin])),
    "Parametrics.extend takes reference object schemas: 'hw.Pin' is a value object schema, "
    + "whose objects are never linked, so that none assigns or binds a parameter");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.Schemas.OfInt])),
    "Parametrics.extend takes reference object schemas: 'Schemas.OfInt' is not one");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Wires])),
    "Parametrics.extend takes reference object schemas: 'hw.Wires' is not one");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, ["nothing"])),
    "Parametrics.extend takes reference object schemas: 'nothing' is not one");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Taken])),
    "Parametrics.extend: 'hw.Taken' already declares a member 'assigned_values'");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Linked])),
    "Parametrics.extend: 'hw.Linked' already declares a member 'bound_objects'");
  assert(JSON.dumps(Plain.ToPlain(hw).Snapshot()) === log); // each refused whole: nothing written
}


<a id="PAR-06"></a>
## PAR-06 · An object assigns a value to a parameter its schema declares, and binds one to an object of any schema, which opts into nothing; a parameter it neither assigns nor binds stays unbound

In [ ]:
hw.transaction(() => {
  const [held] = [...new B.OfObject.Builder(Bus, false, hw).parameters];
  const [width, host] = [...held.params.parameters].map((e: any) => e.parameter);
  const chip = hw.hw.Chip((c: any) => c);
  const bus = hw.hw.Bus((b: any) => b.label("data")
    .assigned_values([(e: any) => e.param(width).value(8n)])
    .bound_objects((a: any) => a.insert((e: any) => e.param(host).obj_ref(chip))));
  const [value] = [...bus.assigned_values];
  const [obj] = [...bus.bound_objects];
  assert(value.param === width && value.value.get() === 8n); // its value an entry property
  assert(obj.param === host && Proxies._durable(obj.obj_ref) === Proxies._durable(chip)); // its object a third link
  const narrow = hw.hw.Bus((b: any) => b.label("narrow").assigned_values([(e: any) => e.param(width).value(1n)]));
  assert([...narrow.bound_objects].length === 0); // host left unbound: it stays unbound
  const loose = hw.hw.Bus((b: any) => b.label("loose"));
  assert([...loose.assigned_values].length === 0 && [...loose.bound_objects].length === 0); // neither is required
  assert([bus, narrow, loose].every((x) => Parametrics.validate(hw, x).length === 0));
  const [chipSchema, spareSchema] = [hw.registered("hw.Chip"), hw.registered("hw.Spare")];
  assert(chipSchema.adjacencies.size === 0 && spareSchema.adjacencies.size === 0 && chipSchema.anonymous.length === 0);
  const spare = hw.hw.Spare((s: any) => s);
  const other = hw.hw.Bus((b: any) => b.label("other").bound_objects((a: any) => a.insert((e: any) => e.param(host).obj_ref(spare))));
  const validate = Validators.Validate(hw);
  assert(validate(Bus, bus).length === 0 && validate(Bus, other).length === 0); // obj_ref links a schema that declares nothing
  const linked = hw.hw.Linked((x: any) => x.bound_objects((a: any) => a.insert((e: any) => e.to(spare))));
  assert(same(validate(hw.registered("hw.Linked"), linked), [ // another relation's link still asks for an adjacency
    "hw.Linked#0.bound_objects[0].to: a 'hw.Spare' cannot fill link 'to'; its schema declares no adjacency, named or "
    + "anonymous, to this relation via 'to'"]));
  const board = hw.hw.Bus((b: any) => b.label("board"));
  const n = hw.References.OfParameter((p: any) => p.name("n")); // a parameter reference in the place of the parameter
  hw.builder("hw.Bus", board).assigned_values([(e: any) => e.param(n).value(3n)]).update(); // what it assigns is unknown here
  assert(Parametrics.validate(hw, board).length === 0);
});


<a id="PAR-07"></a>
## PAR-07 · An assignment or a binding of a parameter its object's schema does not declare refuses its transaction when it commits; validation reports one already held, and a snapshot reads it back as written

In [ ]:
/** hw.Wire, declaring depth, and hw.Top, a singleton declaring n and m. */
function wiring(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.wiring/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Wire: S.OfObject, Top: S.OfObject })
    .spec([
      (_: any) => _.Wire.name("hw.Wire").ref(true).parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        depth: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
      (_: any) => _.Top.name("hw.Top").ref(true).singleton("hw.Top").parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
          m: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
    ])]))]);
}

{
  hw.transaction(wiring);
  hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Wire, t.hw.Top]));
  const log = JSON.dumps(Plain.ToPlain(hw).Snapshot());
  const [depth, free] = hw.transaction(() => {
    const [held] = [...new B.OfObject.Builder(hw.registered("hw.Wire"), false, hw).parameters];
    const [depth] = [...held.params.parameters].map((e: any) => e.parameter);
    const free = hw.Schemas.OfParameter((p: any) => p.for_value([hw.Schemas.OfInt])); // in no set
    return [depth, free];
  });
  raises(ValueError, () => hw.transaction(() => {
    hw.hw.Bus((b: any) => b.assigned_values([(e: any) => e.param(depth).value(8n)])); // a hw.Wire's: written, and
  }), "an entry of 'Parametrics.AssignsValue' assigns a value to parameter 'depth', which 'hw.Bus' " // refused at
    + "does not declare: an object assigns and binds only the parameters its schema declares"); // commit, as a unique clause is
  raises(ValueError, () => hw.transaction(() => {
    hw.hw.Bus((b: any) => b.bound_objects((a: any) => a.insert((e: any) => e.param(free).obj_ref(hw.hw.Chip((c: any) => c)))));
  }), "an entry of 'Parametrics.BindsObject' binds a parameter no set of parameters defines to "
    + "an object, which 'hw.Bus' does not declare");
  raises(ValueError, () => hw.transaction(() => { // an assignment names its parameter
    hw.hw.Bus((b: any) => b.assigned_values((a: any) => a.insert((e: any) => e.value(8n))));
  }), "link 'param' is not set");
  assert(JSON.dumps(Plain.ToPlain(hw).Snapshot()) === log); // each refused whole
  hw.transaction(() => {
    const [held] = [...new B.OfObject.Builder(hw.registered("hw.Top"), false, hw).parameters];
    const [n] = [...held.params.parameters].map((e: any) => e.parameter);
    hw.builder("hw.Top", hw.singleton("hw.Top")).assigned_values([(e: any) => e.param(n).value(3n)]).update();
  });
  hw.transaction(() => { // an object that assigns one already assigns another: the assignment inserted is checked
    const [held] = [...new B.OfObject.Builder(hw.registered("hw.Top"), false, hw).parameters];
    const [, m] = [...held.params.parameters].map((e: any) => e.parameter);
    hw.builder("hw.Top", hw.singleton("hw.Top")).assigned_values((a: any) => a.insert((e: any) => e.param(m).value(4n))).update();
    assert(same([...hw.singleton("hw.Top").assigned_values].map((e: any) => e.value.get()), [3n, 4n]));
  });
  hw.transaction(() => { // the set's declarations removed: the assignments held stay
    const [held] = [...new B.OfObject.Builder(hw.registered("hw.Top"), false, hw).parameters];
    hw.builder("Parametrics.OfParameters", B._linked_data(held.params)).parameters((a: any) => a.clear()).update();
  });
  hw.transaction(() => { // n and m no longer declared: the assignments held are reported, not refused
    assert(same(Parametrics.validate(hw, hw.singleton("hw.Top")), Array(2).fill(
      "an entry of 'Parametrics.AssignsValue' assigns a value to a parameter no set of parameters defines, which "
      + "'hw.Top' does not declare: an object assigns and binds only the parameters its schema declares")));
  });
  const written = Plain.ToPlain(hw).Snapshot(); // the assignments, written as any entries are
  const back: any = new B.Store();
  back.transaction(Parametrics.install);
  Plain.FromPlain(back).Snapshot(written); // read back as written: not refused
  assert(JSON.dumps(Plain.ToPlain(back).Snapshot()) === JSON.dumps(written));
  back.transaction(() => {
    assert(same([...back.singleton("hw.Top").assigned_values].map((e: any) => e.value.get()), [3n, 4n]));
    assert(Parametrics.validate(back, back.singleton("hw.Top")).length === 2);
  });
}


<a id="PAR-08"></a>
## PAR-08 · A schema neither assigns nor binds its parameters: no kind of schema has an adjacency to an assignment or a binding, each links an `object`, and extending one of the meta-schemas is refused

In [ ]:
for (const kind of KINDS) { // each kind holds its set of parameters, and no assignment or binding
  const schema = hw.Schemas[kind].as_schema();
  assert([...schema.adjacencies.values(), ...schema.anonymous].every((a: any) => a.relation !== AssignsValue
    && a.relation !== BindsObject));
}
assert(same([...AssignsValue.entry.links.keys()], ["object", "param"]) && !BindsObject.entry.links.has("defn"));
raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.Schemas.OfObject])),
  "Parametrics.extend: 'Schemas.OfObject' is one of the meta-schemas, whose objects are "
  + "schemas: a schema declares its parameters and neither assigns nor binds them");
assert(!hw.Schemas.OfObject.as_schema().adjacencies.has("assigned_values"));
